# Projeto de Python para Finanças

In [132]:
# instalação
# pip install yfinance pandas numpy plotly nbformat

### Parte 1: Obter cotações e construção de carteira

In [133]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

acoes = ["ITUB4.SA", "PETR4.SA", "VALE3.SA", "IVVB11.SA"]
# IBOVESPA -> ^BVSP
# SP500 -> ^GSPC
# Dólar -> BRL=X
# Ouro (dolarizado) -> GC=F
indices = ["^BVSP", "^GSPC", "BRL=X", "GC=F"]

data_inicio = datetime.now() - timedelta(days=730)
data_inicio = data_inicio.strftime("%Y-%m-%d")
data_fim = datetime.now().strftime("%Y-%m-%d")
print(data_fim, data_inicio)

def pegar_cotacoes(lista_tickers, data_inicio, data_fim):
    df = yf.download(lista_tickers, data_inicio, data_fim, auto_adjust=False)
    df = df["Adj Close"]
    df = df.ffill()
    df = df.dropna()
    return df

lista_tickers = acoes + indices
df_cotacoes = pegar_cotacoes(lista_tickers, data_inicio, data_fim)
display(df_cotacoes)

[*********************100%***********************]  8 of 8 completed

2026-02-12 2024-02-13


Ticker,BRL=X,GC=F,ITUB4.SA,IVVB11.SA,PETR4.SA,VALE3.SA,^BVSP,^GSPC
Date,,,,,,,,
2024-02-14,4.9520,1990.300049,24.846970,275.850006,30.045233,53.147747,127018.0,5000.620117
2024-02-15,4.9683,2002.099976,25.057108,277.700012,31.005449,52.977921,127804.0,5029.729980
2024-02-16,4.9724,2011.500000,25.028124,276.350006,31.291317,54.991581,128726.0,5005.569824
2024-02-19,4.9651,2011.500000,25.303478,276.700012,31.445250,54.579144,129036.0,5005.569824
2024-02-20,4.9564,2027.500000,25.854179,272.299988,31.115395,53.382271,129916.0,4975.509766
...,...,...,...,...,...,...,...,...
2026-02-05,5.2380,4861.399902,45.520000,402.920013,37.000000,86.449997,182127.0,6798.399902
2026-02-06,5.2705,4951.200195,46.750000,406.500000,36.650002,85.629997,182950.0,6932.299805
2026-02-09,5.2159,5050.899902,48.310001,406.299988,37.320000,87.309998,186241.0,6964.819824


In [134]:
# carteira em termos de valores financeiros (R$)
dic_carteira = {
    "ITUB4.SA": 5000,
    "VALE3.SA": 3000,
    "PETR4.SA": 4000,
    "IVVB11.SA": 6000
}

df_carteira = pd.DataFrame()
total_investido = sum(dic_carteira.values())
print(total_investido)

for ativo in dic_carteira:
    preco_inicial_ativo = df_cotacoes[ativo].iloc[0]
    qtde_acoes = dic_carteira[ativo] / preco_inicial_ativo
    df_carteira[ativo] = df_cotacoes[ativo] * qtde_acoes

df_carteira["Total"] = df_carteira.sum(axis=1)
display(df_carteira)



18000


,ITUB4.SA,VALE3.SA,PETR4.SA,IVVB11.SA,Total
Date,,,,,
2024-02-14,5000.000000,3000.000000,4000.000000,6000.000000,18000.000000
2024-02-15,5042.286509,2990.413902,4127.836124,6040.239392,18200.775927
2024-02-16,5036.453993,3104.077822,4165.894433,6010.875476,18317.301724
2024-02-19,5091.864047,3080.797206,4186.387877,6018.488441,18377.537571
2024-02-20,5202.682620,3013.238027,4142.473427,5922.783725,18281.177799
...,...,...,...,...,...
2026-02-05,9160.070862,4879.792753,4925.906253,8763.893518,27729.663387
2026-02-06,9407.585863,4833.506707,4879.310046,8841.761631,27962.164248
2026-02-09,9721.507722,4928.336708,4968.508645,8837.411176,28455.764250


### Parte 2: Rentabilidade e Comparação com Benchmarks

In [135]:
df_cotacoes["SP500 (R$)"] = df_cotacoes["^GSPC"] * df_cotacoes["BRL=X"]
df_cotacoes["Ouro (R$)"] = df_cotacoes["GC=F"] * df_cotacoes["BRL=X"]
df_cotacoes["Dólar"] = df_cotacoes["BRL=X"]

df_cotacoes = df_cotacoes.drop(columns=["BRL=X", "GC=F", "^GSPC"])
display(df_cotacoes)

Ticker,ITUB4.SA,IVVB11.SA,PETR4.SA,VALE3.SA,^BVSP,SP500 (R$),Ouro (R$),Dólar
Date,,,,,,,,
2024-02-14,24.846970,275.850006,30.045233,53.147747,127018.0,24763.071526,9855.966123,4.9520
2024-02-15,25.057108,277.700012,31.005449,52.977921,127804.0,24989.206787,9947.033040,4.9683
2024-02-16,25.028124,276.350006,31.291317,54.991581,128726.0,24889.696337,10001.982979,4.9724
2024-02-19,25.303478,276.700012,31.445250,54.579144,129036.0,24853.153791,9987.298271,4.9651
2024-02-20,25.854179,272.299988,31.115395,53.382271,129916.0,24660.616192,10049.100833,4.9564
...,...,...,...,...,...,...,...,...
2026-02-05,45.520000,402.920013,37.000000,86.449997,182127.0,35610.018118,25464.012280,5.2380
2026-02-06,46.750000,406.500000,36.650002,85.629997,182950.0,36536.687390,26095.301536,5.2705
2026-02-09,48.310001,406.299988,37.320000,87.309998,186241.0,36327.803333,26344.988519,5.2159


In [136]:
# valor_final = 18102
# valor_inicial = 18000
# variacao_reais = valor_final - valor_inicial
# print(variacao_reais)

# variacao_percentual = valor_final / valor_inicial - 1
# print(variacao_percentual)

def calcular_retorno(df):
    retorno = df.iloc[-1] / df.iloc[0] - 1
    return retorno

display(calcular_retorno(df_carteira))
display(calcular_retorno(df_cotacoes))

ITUB4.SA     0.986963
VALE3.SA     0.695086
PETR4.SA     0.267422
IVVB11.SA    0.466195
Total        0.604830
dtype: float64

Ticker
ITUB4.SA      0.986963
IVVB11.SA     0.466195
PETR4.SA      0.267422
VALE3.SA      0.695086
^BVSP         0.493481
SP500 (R$)    0.456575
Ouro (R$)     1.673817
Dólar         0.049313
dtype: float64

In [137]:
df_comparacao = df_cotacoes.drop(columns=acoes)
df_comparacao["Carteira"] = df_carteira["Total"]
display(df_comparacao)
print(calcular_retorno(df_comparacao))

Ticker,^BVSP,SP500 (R$),Ouro (R$),Dólar,Carteira
Date,,,,,
2024-02-14,127018.0,24763.071526,9855.966123,4.9520,18000.000000
2024-02-15,127804.0,24989.206787,9947.033040,4.9683,18200.775927
2024-02-16,128726.0,24889.696337,10001.982979,4.9724,18317.301724
2024-02-19,129036.0,24853.153791,9987.298271,4.9651,18377.537571
2024-02-20,129916.0,24660.616192,10049.100833,4.9564,18281.177799
...,...,...,...,...,...
2026-02-05,182127.0,35610.018118,25464.012280,5.2380,27729.663387
2026-02-06,182950.0,36536.687390,26095.301536,5.2705,27962.164248
2026-02-09,186241.0,36327.803333,26344.988519,5.2159,28455.764250


Ticker
^BVSP         0.493481
SP500 (R$)    0.456575
Ouro (R$)     1.673817
Dólar         0.049313
Carteira      0.604830
dtype: float64


In [138]:
df_comparacao = (df_comparacao / df_comparacao.iloc[0] - 1 ) * 100
display(df_comparacao)

import plotly.express as px

grafico = px.line(df_comparacao, x=df_comparacao.index, y=df_comparacao.columns)
grafico.update_layout(template="plotly_dark")
grafico.show()

Ticker,^BVSP,SP500 (R$),Ouro (R$),Dólar,Carteira
Date,,,,,
2024-02-14,0.000000,0.000000,0.000000,0.000000,0.000000
2024-02-15,0.618810,0.913196,0.923978,0.329154,1.115422
2024-02-16,1.344691,0.511345,1.481507,0.411956,1.762787
2024-02-19,1.588751,0.363777,1.332514,0.264533,2.097431
2024-02-20,2.281566,-0.413742,1.959572,0.088848,1.562099
...,...,...,...,...,...
2026-02-05,43.386764,43.802913,158.361402,5.775440,54.053685
2026-02-06,44.034704,47.545055,164.766551,6.431745,55.345357
2026-02-09,46.625675,46.701524,167.299909,5.329156,58.087579


### Parte 3: Análise de Risco

In [146]:
import numpy as np

# correlação
df_cotacoes["Carteira"] = df_carteira["Total"]

# rentabilidade diária -> função logaritmica
tabela_rentabilidade_diaria = df_cotacoes / df_cotacoes.shift(1)
tabela_rentabilidade_diaria = np.log(tabela_rentabilidade_diaria).dropna()
tabela_correlacao = tabela_rentabilidade_diaria.corr()

grafico_correlacao = px.imshow(tabela_correlacao, text_auto=True, color_continuous_scale="greens")
grafico_correlacao.update_layout(template="plotly_dark")
grafico_correlacao.show()

# variância dos retornos diários do ativo (aplicando a função logarítmica)

tabela_volatilidade = tabela_rentabilidade_diaria.std() * np.sqrt(252)

display(tabela_volatilidade)

Ticker
ITUB4.SA      0.198037
IVVB11.SA     0.156190
PETR4.SA      0.239537
VALE3.SA      0.231668
^BVSP         0.143763
SP500 (R$)    0.207728
Ouro (R$)     0.243052
Dólar         0.127946
Carteira      0.110815
dtype: float64

### Parte 4: Análise Técnica e Indicadores

In [155]:
ticker = "VALE3.SA"

import yfinance as yf
import plotly.graph_objects as go

df = yf.download(ticker, "2020-01-01", "2025-12-31", multi_level_index=False)
display(df)

# média movel de 50 dias
df["MM50"] = df["Close"].rolling(50).mean()

# média móvel de 200 dias
df["MM200"] = df["Close"].rolling(200).mean()


grafico = go.Figure()

# adicionar as series do gráfico que você quer (os subgráficos)
grafico.add_trace(go.Candlestick(x=df.index, open=df["Open"], close=df["Close"], high=df["High"], 
                                 low=df["Low"], name="Price"))

# adicionar a serie da MM50
grafico.add_trace(go.Scatter(x=df.index, y=df["MM50"], name="MM50", line={"color": "blue", "width": 1}))

# adicionar a serie da MM200
grafico.add_trace(go.Scatter(x=df.index, y=df["MM200"], name="MM200", line={"color": "yellow", "width": 1}))

grafico.update_layout(template="plotly_dark")
grafico.show()




[*********************100%***********************]  1 of 1 completed


,Close,High,Low,Open,Volume
Date,,,,,
2020-01-02,30.140167,30.201189,29.818405,29.946000,17509700
2020-01-03,29.918257,30.234470,29.724091,29.779567,17284800
2020-01-06,29.740732,29.846136,29.485543,29.846136,32787800
2020-01-07,29.957090,30.062494,29.624235,29.679710,16326400
2020-01-08,29.962631,30.162345,29.746275,30.068037,15298500
...,...,...,...,...,...
2025-12-22,72.919998,73.370003,70.809998,71.029999,27853000
2025-12-23,72.900002,73.529999,72.580002,73.230003,17140900
2025-12-26,73.120003,73.349998,72.620003,72.860001,16368800
